In [2]:
from datasets import load_dataset
from tqdm import tqdm
from ir_datasets_longeval import load
import pandas as pd
import gzip
import json

In [4]:
qid_to_docs = {}

for i in tqdm(load_dataset("gijshendriksen/LongEval", split="train")):
    if i["query_id"] not in qid_to_docs:
        qid_to_docs[i["query_id"]] = {}
    score = qid_to_docs[i["query_id"]].get(i["id"], {}).get("score", 0)

    qid_to_docs[i["query_id"]][i["id"]] = {"id": i["id"], "title": i["title"], "rank": i["rank"] + score}


100%|██████████| 48738/48738 [00:04<00:00, 10206.81it/s]


In [13]:
df = []

for dataset in load("longeval-sci/clef-2025-test").get_datasets():
    docs_store = dataset.docs_store()
    for query in dataset.queries_iter():
        if query.query_id not in qid_to_docs:
            print(f"skip {query.query_id}...")
            continue

        for doc_id in qid_to_docs[query.query_id]:
            try:
                irds_doc = docs_store.get(str(doc_id))

                df.append({
                    "query_id": query.query_id,
                    "snapshot": dataset.get_snapshot(),
                    "query": query.default_text(),
                    "Rank": qid_to_docs[query.query_id][doc_id]["rank"],
                    "doc_id": doc_id,
                    "Title (Core)": qid_to_docs[query.query_id][doc_id]["title"],
                    "Title (LongEval)": irds_doc.title
                })
            except:
                pass

df = pd.DataFrame(df)

skip 1f841355-4fee-4949-9993-8988f12743cc...
skip 2ce89e34-6c38-4c3d-a4e8-98c5a2ca539b...


In [14]:
df

,query_id,snapshot,query,Rank,doc_id,Title (Core),Title (LongEval)
0,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,7,890363,Measurement of the Zero Crossing in a Feshbach...,Measurement of the Zero Crossing in a Feshbach...
1,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,4,31283470,FPGA-based implementation of the back-EMF symm...,FPGA-based implementation of the back-EMF symm...
2,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,0,45156357,Zero-Crossing Statistics for Non-Markovian Tim...,Zero-Crossing Statistics for Non-Markovian Tim...
3,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,5,7431333,Differential branching fraction and angular an...,Differential branching fraction and angular an...
4,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,12,860055,Nonlinear Cosmological Power Spectra in Real a...,Nonlinear Cosmological Power Spectra in Real a...
...,...,...,...,...,...,...,...
9241,1a0b576f-7e47-4010-9c10-44393c49fa08,2025-01,surface structure,2,17026331,On the surface structure of sunspots,On the surface structure of sunspots
9242,1a0b576f-7e47-4010-9c10-44393c49fa08,2025-01,surface structure,3,78560738,Elucidating Surface Structure with Action Spec...,Elucidating Surface Structure with Action Spec...
9243,1a0b576f-7e47-4010-9c10-44393c49fa08,2025-01,surface structure,5,906273,Microscopic Surface Structure of Liquid Alkali...,Microscopic Surface Structure of Liquid Alkali...
9244,1a0b576f-7e47-4010-9c10-44393c49fa08,2025-01,surface structure,9,41535572,Local surface structure and composition contro...,Local surface structure and composition contro...


In [15]:
df.head(25)

,query_id,snapshot,query,Rank,doc_id,Title (Core),Title (LongEval)
0,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,7,890363,Measurement of the Zero Crossing in a Feshbach...,Measurement of the Zero Crossing in a Feshbach...
1,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,4,31283470,FPGA-based implementation of the back-EMF symm...,FPGA-based implementation of the back-EMF symm...
2,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,0,45156357,Zero-Crossing Statistics for Non-Markovian Tim...,Zero-Crossing Statistics for Non-Markovian Tim...
3,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,5,7431333,Differential branching fraction and angular an...,Differential branching fraction and angular an...
4,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,12,860055,Nonlinear Cosmological Power Spectra in Real a...,Nonlinear Cosmological Power Spectra in Real a...
5,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,1,861374,Fluxtube model atmospheres and Stokes V zero-c...,Fluxtube model atmospheres and Stokes V zero-c...
6,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,2,24815400,On the zero crossing of the three-gluon vertex,On the zero crossing of the three-gluon vertex
7,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,3,4469258,Automatic carrier acquisition system for phase...,Automatic carrier acquisition system for phase...
8,d585f080-4519-4952-8278-0d13fcd03fec,2024-11,zero-crossing,12,24885299,Frequency stability review,Frequency stability review
9,254ecdb5-45e5-45ce-9b3f-492d1e1c085e,2024-11,dsrna,0,78227424,Effects of Nanoparticles on Double-Stranded RN...,Effects of Nanoparticles on Double-Stranded RN...


In [16]:
df.to_json("top-core-documents.jsonl.gz", lines=True, orient="records")